# Thermomaps Ising — portable working notebook

Run with the isolated Ising environment, in cell order. Original executed outputs are in `records/`. Low-temperature sampling is not yet validated.


Install `requirements.txt` in your Ising environment before opening this notebook. This notebook does not clone or install packages.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sys, random, json
ROOT = next((p for p in (Path.cwd(), Path.cwd() / "ising")
             if (p / "thermomaps-root" / "ising" / "base.py").is_file()), None)
assert ROOT is not None, "Open this notebook from the repository root or ising folder."
ROOT = ROOT.resolve()
SOURCE = ROOT / "thermomaps-root"
# Avoid silently reusing ExpTM's different tm package in this kernel.
for name in ("tm", "ising", "data"):
    mod = sys.modules.get(name)
    if mod is not None and getattr(mod, "__file__", None):
        assert SOURCE in Path(mod.__file__).resolve().parents, "Restart using the Ising kernel."
sys.path.insert(0, str(SOURCE))
import numpy as np
import torch
random.seed(42); np.random.seed(42); torch.manual_seed(42)
torch.set_num_threads(2)
RUN_DIR = ROOT / "runs" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR.mkdir(parents=True, exist_ok=False)
EPOCHS = 20
GENERATION_SAMPLES = 5000
GENERATION_BATCH = 50
assert GENERATION_SAMPLES % GENERATION_BATCH == 0
(RUN_DIR / "settings.json").write_text(json.dumps({
    "epochs": EPOCHS, "generation_samples": GENERATION_SAMPLES,
    "generation_batch": GENERATION_BATCH, "torch": torch.__version__,
    "numpy": np.__version__, "python": sys.version, "seed": 42,
}, indent=2))
print("Python:", sys.executable)
print("Run outputs:", RUN_DIR)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from ising.observables import Energy, Magnetization
from ising.samplers import SwendsenWangSampler, SingleSpinFlipSampler
from ising.base import IsingModel

from data.trajectory import EnsembleTrajectory, MultiEnsembleTrajectory
from data.dataset import MultiEnsembleDataset
from data.generic import Summary

from tm.core.prior import GlobalEquilibriumHarmonicPrior, UnitNormalPrior
from tm.core.backbone import ConvBackbone
from tm.core.diffusion_model import DiffusionTrainer, SteeredDiffusionSampler
from tm.core.diffusion_process import VPDiffusion
from tm.architectures.unet_2d_mid_attn import Unet2D


In [ ]:
np.random.seed(42)

IM = IsingModel(
    sampler=SingleSpinFlipSampler,
    size=8,
    warmup=1000,
    temp=4,
    Jx=1,
    Jy=1,
)

IM.simulate(
    steps=100,
    observables=[Energy(Jx=1, Jy=1), Magnetization()],
    sampling_frequency=1,
)

print("Simulation completed")

First we simulate the Ising model over a range of temperatures, recording a `trajectory` consisting of lattice configurations. These trajectories are then aggregated into a `MultiEnsembleDataset` (i.e. many trajectories gathered under different thermodynamic ensembles).

In [ ]:
frames = np.asarray(IM.trajectory.coordinates)
energy = np.asarray(IM.trajectory.observables["energy"].quantity)
magnetization = np.asarray(
    IM.trajectory.observables["magnetization"].quantity
)

print("Configurations:", frames.shape)
print("Spin values:", np.unique(frames))
print("Energy shape:", energy.shape)
print("Magnetization shape:", magnetization.shape)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

axes[0].imshow(
    frames[-1], cmap="binary", vmin=-1, vmax=1,
    interpolation="nearest"
)
axes[0].set_title("Last configuration, T=4")

axes[1].plot(energy / (IM.size ** 2))
axes[1].set_title("Energy per spin")
axes[1].set_xlabel("Saved frame")

axes[2].plot(magnetization)
axes[2].set_title("Absolute magnetization |m|")
axes[2].set_xlabel("Saved frame")
axes[2].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

temperatures = np.round(np.arange(0.3, 3.5, 0.2), 1)
trajectories = []

for temperature in tqdm(temperatures, desc="Generating Ising data"):
    IM = IsingModel(
        sampler=SingleSpinFlipSampler,
        size=8,
        warmup=1000,
        temp=float(temperature),
        Jx=1,
        Jy=1,
    )

    IM.simulate(
        steps=5000,
        observables=[Energy(Jx=1, Jy=1), Magnetization()],
        sampling_frequency=1,
    )

    trajectories.append(IM.trajectory)

dataset = MultiEnsembleDataset(trajectories, Summary())

print("Temperatures:", temperatures)
print("Number of trajectories:", len(trajectories))
print("First trajectory shape:",
      np.asarray(trajectories[0].coordinates).shape)


Each of the trajectories has a `Summary` object attached to it, which is just a `dict` containing metadata about the trajectory. If the summaries associated with all the trajectories in the `MultiEnsembleDataset` all have the same keys, the contents of the `MultiEnsembleDataset` can be conveniently cast as a pandas dataframe.

In [ ]:
df = dataset.to_dataframe()
df

The dataframe representation is especially powerful for creating subsets of a `MultiEnsembleDataset`. In this cell we extract a sub-dataframe containing trajectories at temperatures `T=1.5` and `T=2.5`. The sub-dataset can be initialized from the restricted dataframe.

**Our training data is going to be 5,000 samples from two temperatures (total of 10,000 samples) asymetrically spaced about the critical temperature $T_c \approx 2.26$**

In [ ]:
sub_df = df[df['temperature'].isin([1.5,2.5])]
sub_dataset = dataset.from_dataframe(sub_df)
sub_df

Here we extract the relevant observables for the ising model from each temperature in the dataset and plot the distribution of magnetization and energy as a function of temperature.

In [ ]:
simulated_M_v_T_mean = {t.state_variables['temperature']: t.observables['magnetization'].quantity.mean() for t in dataset.trajectories}
simulated_E_v_T_mean = {t.state_variables['temperature']: t.observables['energy'].quantity.mean() for t in dataset.trajectories}
simulated_M_v_T_std = {t.state_variables['temperature']: t.observables['magnetization'].quantity.std() for t in dataset.trajectories}
simulated_E_v_T_std = {t.state_variables['temperature']: t.observables['energy'].quantity.std() for t in dataset.trajectories}

train_M_v_T = {traj.state_variables['temperature']: simulated_M_v_T_mean[traj.state_variables['temperature']] for traj in sub_dataset.trajectories}
train_E_v_T = {traj.state_variables['temperature']: simulated_E_v_T_mean[traj.state_variables['temperature']] for traj in sub_dataset.trajectories}

In [ ]:
# Vary these to view different trajectories/configurations
trajectory_idx = 0
frame = -1

default_blue = plt.rcParams['axes.prop_cycle'].by_key()['color'][0]
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4), dpi=150)
ax1.plot(list(simulated_M_v_T_mean.keys()), list(simulated_M_v_T_mean.values()), c=default_blue)
ax1.scatter(list(simulated_M_v_T_mean.keys()), list(simulated_M_v_T_mean.values()), c=default_blue)
ax1.scatter(list(train_M_v_T.keys()), list(train_M_v_T.values()),
            c='r', marker='*', edgecolors='k', s=200, zorder=2, label='Train Data')
ax1.fill_between(list(simulated_M_v_T_mean.keys()),
                 [max(m - s, 0) for m, s in zip(simulated_M_v_T_mean.values(), simulated_M_v_T_std.values())],
                 [min(m + s, 1) for m, s in zip(simulated_M_v_T_mean.values(), simulated_M_v_T_std.values())],
                 alpha=0.2, color=default_blue)
ax1.set_ylabel('Magnetization')
ax1.set_xlabel('Temperature')


ax2.plot(list(simulated_E_v_T_mean.keys()), list(simulated_E_v_T_mean.values()), c=default_blue)
ax2.scatter(list(simulated_E_v_T_mean.keys()), list(simulated_E_v_T_mean.values()), c=default_blue)
ax2.scatter(list(train_E_v_T.keys()), list(train_E_v_T.values()),
            c='r', marker='*', edgecolors='k', s=200, zorder=2, label='Train Data')
ax2.fill_between(list(simulated_E_v_T_mean.keys()),
                 [m - s for m, s in zip(simulated_E_v_T_mean.values(), simulated_E_v_T_std.values())],
                 [m + s for m, s in zip(simulated_E_v_T_mean.values(), simulated_E_v_T_std.values())],
                 alpha=0.2, color=default_blue)
ax2.set_ylabel('Energy')
ax2.set_xlabel('Temperature')

img = ax3.imshow(
    trajectories[trajectory_idx].coordinates[frame],
    aspect='equal',
    cmap='binary',
    vmin=-1,
    vmax=1
)
ax3.set_title(f'Frame {range(len(trajectories[trajectory_idx].coordinates))[frame]} of sim. traj. at T = {trajectories[trajectory_idx].state_variables["temperature"]}')
ax3.set_xticks([])
ax3.set_yticks([])

ax1.legend()
ax2.legend()
plt.tight_layout()
plt.show()


In [ ]:
traj = trajectories[0]  # T = 0.3
configs = np.asarray(traj.coordinates)
energies = np.asarray(traj.observables["energy"].quantity)

print("Number of unique configurations:", np.unique(configs, axis=0).shape[0])
print("Energy range:", energies.min(), energies.max())

In [ ]:
# Inspect the existing trajectory at T=0.3
low_traj = next(
    traj for traj in dataset.trajectories
    if np.isclose(traj.state_variables["temperature"], 0.3)
)

low_frames = np.asarray(low_traj.coordinates)
low_m = np.asarray(low_traj.observables["magnetization"].quantity)
low_e = np.asarray(low_traj.observables["energy"].quantity)

fig, axes = plt.subplots(1, 4, figsize=(14, 3))

for ax, idx in zip(axes[:2], [0, -1]):
    ax.imshow(
        low_frames[idx], cmap="binary",
        vmin=-1, vmax=1, interpolation="nearest"
    )
    ax.set_title("First saved frame" if idx == 0 else "Last saved frame")

axes[2].plot(low_m)
axes[2].set_title("|m| at T=0.3")
axes[2].set_xlabel("Saved frame")
axes[2].set_ylim(0, 1.05)

axes[3].plot(low_e)
axes[3].set_title("Total energy at T=0.3")
axes[3].set_xlabel("Saved frame")

plt.tight_layout()
plt.show()
print(
    "Distinct saved configurations:",
    np.unique(low_frames.reshape(len(low_frames), -1), axis=0).shape[0]
)

Loaders for ThermoMap data can also be constructed from a `MultiEnsembleDataset`. The coordinates of the trajectory object are joined with the specified `state_variables` along the channel dimension.

In [ ]:
train_loader, test_loader = sub_dataset.to_TMLoader(train_size=0.8, test_size=0.2, state_variables=['temperature'])

In [ ]:
print("Training data shape:", train_loader.data.shape)
print("Test data shape:", test_loader.data.shape)

Here we initialize the components of the ThermoMap algorithm.

First, we initialize the prior distribution that samples from the dataset will be mapped onto. The `GlobalEquilibriumHarmonicPrior` (`GEHP`) implements the temperature scaling at the core of the ThermoMap algorithm. We view the prior distribution as the equilibrium distribution arising from dynamics of a *physical* prior system. The `GEHP` assumes that the prior system is a particle in a harmonic trap, which results in a Normal equilibrium distribution whose variance is proportional to temperature.

I have also provided a `UnitNormalPrior` (`UNP`) which is a drop-in replacement for the `GEHP` that does not include any temperature rescaling. As such, a ThermoMap trained with a `UNP` is a standard diffusion model.

The prior requires the shape of the training data as input as well as the channels that are associated with coordinates and fluctuations (i.e. temperatures). The coordinate dimensions are sampled from a Normal distribution with variance determined by the fluctuation dimensions, while the fluctuation channels are sampled from a unit Normal distribution.

In [ ]:
prior = GlobalEquilibriumHarmonicPrior(shape=train_loader.data.shape, channels_info={"coordinate": [0], "fluctuation": [1]})

The `model` is the black-box that is used to parameterize the score. Here we opt for a 2D U-net with attention at the upsampling/downsampling bottleneck.

In [ ]:
model = Unet2D(dim=16, dim_mults=(1, 2, 4), resnet_block_groups=8, channels=2)

The `backbone` is a wrapper around the model which contains the optimizer, scheduler, and other utilities and implements saving and loading the `model`.

In [ ]:
backbone = ConvBackbone(model=model,
                        data_shape=train_loader.data_dim,
                        target_shape=8,
                        num_dims=4,
                        lr=1e-3,
                        eval_mode="train")

The `diffusion` instantiates the noise schedule, and implements the forward and reverse transition kernels of the diffusion process. Functionally, the `diffusion` implements the Euler-Maryuma SDE/Euler ODE solver for the variance-preserving diffusion process.

In [ ]:
diffusion = VPDiffusion(num_diffusion_timesteps=100)

The `trainer` implements the training algorithm, consisting of sampling noise from the `prior`, calls to the transition kernels of the `diffusion`, and parameterizing the `backbone` to match the score.

In [ ]:
trainer = DiffusionTrainer(diffusion,
                           backbone,
                           train_loader,
                           prior,
                           model_dir=str(RUN_DIR / "models"), # save models every epoch
                           pred_type="noise", # set to "noise" or "x0"
                           test_loader=test_loader # optional
                           )

In [ ]:
trainer.train(EPOCHS, loss_type="smooth_l1", batch_size=8)
# Note that the test loss is usually slightly lower than the training loss. This is because
# the training loss is averaged over each epoch (which contains many updates to the weights
# via backprop) while the test loss is evaluated at the end of each epoch. Is there a
# better way to do this? Probably. But it's low priority at the moment.

The `sampler` is similar to the `trainer`, but rather than calling the $p(x_0|x_t)$ kernel, the `sampler` iteratively calls the $p(x_{t-1}|x_t)$ kernel.

In [ ]:
sampler = SteeredDiffusionSampler(diffusion,
                                  backbone,
                                  train_loader,
                                  prior,
                                  pred_type='noise', # must be the same as in DiffusionTrainer
                                  )

With our trained model, we generated samples at a range of different temperatures, store the coordinates in an `EnsembleTrajectory` object, and evaluate observables over the coordinates of the trajectory. Note that there is no temporal order between points in the trajectory, as was the case of the Ising simulator.

In [ ]:
trajectories = []
pbar = tqdm(np.arange(0.3, 3.5, 0.2))
for temperature in pbar:
  pbar.set_description(f"Generating at T={temperature:.1f}")
  samples = sampler.sample_loop(num_samples=GENERATION_SAMPLES, batch_size=GENERATION_BATCH, temperature=temperature)
  coords = samples[:,0,:,:].detach().cpu().numpy() # take coordinate dimension
  # binarize
  coords[coords > 0] = 1
  coords[coords < 0] = -1

  # store in trajectory
  trajectory = EnsembleTrajectory(summary=Summary(info="Generated trajectory"),
                                  state_variables=Summary(temperature=temperature),
                                  coordinates=coords)

  # evaluate observables over trajectory coordinates and add to trajectory object
  energy = Energy()
  energy.evaluate(trajectory.coordinates)

  mag = Magnetization()
  mag.evaluate(trajectory.coordinates)

  trajectory.add_observable(energy)
  trajectory.add_observable(mag)
  trajectories.append(trajectory)

In [ ]:
generated_M_v_T_mean = {t.state_variables['temperature']: t.observables['magnetization'].quantity.mean() for t in trajectories}
generated_E_v_T_mean = {t.state_variables['temperature']: t.observables['energy'].quantity.mean() for t in trajectories}
generated_M_v_T_std = {t.state_variables['temperature']: t.observables['magnetization'].quantity.std() for t in trajectories}
generated_E_v_T_std = {t.state_variables['temperature']: t.observables['energy'].quantity.std() for t in trajectories}

We find remarkable agreement overlaying the $M(T)$ and $E(T)$ dependence of the ThermoMap-generated and simulated samples.

In [ ]:
# Vary these to view different trajectories/configurations
trajectory_idx = -1
frame = -1

default_blue = plt.rcParams['axes.prop_cycle'].by_key()['color'][0]
default_orange = plt.rcParams['axes.prop_cycle'].by_key()['color'][1]

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4), dpi=150)

# Plot the simulated ising data
ax1.plot(list(simulated_M_v_T_mean.keys()), list(simulated_M_v_T_mean.values()), c=default_blue, marker='o', label="MC-simulated")
ax1.fill_between(list(simulated_M_v_T_mean.keys()),
                 [max(m - s, 0) for m, s in zip(simulated_M_v_T_mean.values(), simulated_M_v_T_std.values())],
                 [min(m + s, 1) for m, s in zip(simulated_M_v_T_mean.values(), simulated_M_v_T_std.values())],
                 alpha=0.2, color=default_blue)

# Plot the generated ising data
ax1.plot(list(generated_M_v_T_mean.keys()), list(generated_M_v_T_mean.values()), c=default_orange, marker='o', label="TM-simulated")
ax1.fill_between(list(generated_M_v_T_mean.keys()),
                 [max(m - s, 0) for m, s in zip(generated_M_v_T_mean.values(), generated_M_v_T_std.values())],
                 [min(m + s, 1) for m, s in zip(generated_M_v_T_mean.values(), generated_M_v_T_std.values())],
                 alpha=0.2, color=default_orange)

# Plot the training data
ax1.scatter(list(train_M_v_T.keys()), list(train_M_v_T.values()), c='r', marker='*', edgecolors='k', s=200, zorder=2, label='Train Data')

ax1.set_ylabel('Magnetization')
ax1.set_xlabel('Temperature')

# Plot the simulated ising data
ax2.plot(list(simulated_E_v_T_mean.keys()), list(simulated_E_v_T_mean.values()), c=default_blue, marker='o', label="MC-simulated")
ax2.fill_between(list(simulated_E_v_T_mean.keys()),
                 [m - s for m, s in zip(simulated_E_v_T_mean.values(), simulated_E_v_T_std.values())],
                 [m + s for m, s in zip(simulated_E_v_T_mean.values(), simulated_E_v_T_std.values())],
                 alpha=0.2, color=default_blue)

# Plot the generated ising data
ax2.plot(list(generated_E_v_T_mean.keys()), list(generated_E_v_T_mean.values()), c=default_orange, marker='o', label="TM-simulated")
ax2.fill_between(list(generated_E_v_T_mean.keys()),
                 [m - s for m, s in zip(generated_E_v_T_mean.values(), generated_E_v_T_std.values())],
                 [m + s for m, s in zip(generated_E_v_T_mean.values(), generated_E_v_T_std.values())],
                 alpha=0.2, color=default_orange)

# Plot the training data
ax2.scatter(list(train_E_v_T.keys()), list(train_E_v_T.values()), c='r', marker='*', edgecolors='k', s=200, zorder=2, label='Train Data')

ax2.set_ylabel('Energy')
ax2.set_xlabel('Temperature')

img = ax3.imshow(trajectories[trajectory_idx].coordinates[frame], aspect='equal', cmap='binary')
ax3.set_title(f'Frame {range(len(trajectories[trajectory_idx].coordinates))[frame]} of gen. traj. at T = {round(trajectories[trajectory_idx].state_variables["temperature"], 2)}')
ax3.set_xticks([])
ax3.set_yticks([])

ax1.legend()
ax2.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4), dpi=150)

ax1.plot(list(simulated_M_v_T_mean.keys()),
        -np.gradient(np.array(list(simulated_M_v_T_mean.values())), list(simulated_M_v_T_mean.keys())),
        c=default_blue, marker='o', label="MC")

ax1.plot(list(generated_M_v_T_mean.keys()),
        -np.gradient(np.array(list(generated_M_v_T_mean.values())), list(generated_M_v_T_mean.keys())),
        c=default_orange, marker='o', label="TM")

ax2.plot(list(simulated_E_v_T_mean.keys()),
        np.gradient(np.array(list(simulated_E_v_T_mean.values()))/8**2, list(simulated_E_v_T_mean.keys())),
        c=default_blue, marker='o', label="MC")

ax2.plot(list(generated_E_v_T_mean.keys()),
        np.gradient(np.array(list(generated_E_v_T_mean.values()))/8**2, list(generated_E_v_T_mean.keys())),
        c=default_orange, marker='o', label="TM")

ax1.set_xlabel("Temperature")
ax1.set_ylabel(fr"$-\partial M/\partial T$")
ax1.legend()

ax2.set_xlabel("Temperature")
ax2.set_ylabel(fr"$C/k_BN = \partial E/\partial T$")
ax2.legend()


### Numerical derivatives: low-temperature sampling caveat

At T = 0.3, all 5,000 saved MC configurations were identical,
with |m| = 0.5 and total energy E = -96. The configuration
contains a stripe, consistent with trapping under single-spin-flip
sampling. This trajectory is not a reliable equilibrium reference.

Including this point produces spurious low-temperature derivatives
and also affects the derivative at the neighboring temperature.

For this supplementary comparison, T = 0.3 is excluded from both
MC and TM curves before numerical differentiation. The original
data and plots are retained; no retraining or resampling is performed.

The remaining MC reference points have not undergone a full
equilibration assessment. The derivative peaks are finite-size,
discrete-grid estimates, not precise critical-temperature estimates.

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(9, 4), dpi=150)

series = [
    ("MC", simulated_M_v_T_mean, simulated_E_v_T_mean),
    ("TM", generated_M_v_T_mean, generated_E_v_T_mean),
]

for label, m_dict, e_dict in series:
    temps = np.array(sorted(m_dict))
    keep = ~np.isclose(temps, 0.3)
    temps = temps[keep]

    mean_m = np.array([m_dict[t] for t in temps])
    mean_e = np.array([e_dict[t] for t in temps]) / 64

    axes[0].plot(
        temps, -np.gradient(mean_m, temps), "o-", label=label
    )
    axes[1].plot(
        temps, np.gradient(mean_e, temps), "o-", label=label
    )

axes[0].set_ylabel(r"$-d\langle |m| \rangle/dT$")
axes[1].set_ylabel(r"$d(\langle E\rangle/N)/dT$")

for ax in axes:
    ax.set_xlabel("Temperature")
    ax.legend()

fig.suptitle("T = 0.3 excluded before differentiation")
plt.tight_layout()
plt.show()

In [ ]:
# Save generated coordinates and temperature labels without overwriting older runs.
np.savez_compressed(
    RUN_DIR / "generated_samples.npz",
    temperatures=np.array([t.state_variables["temperature"] for t in trajectories]),
    coordinates=np.stack([np.asarray(t.coordinates) for t in trajectories]),
)
print("Saved to", RUN_DIR)
